In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.base import clone

warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Data paths
TRAIN_PATH = "../data/in/cattle_data_train.csv"
TARGET_FEATURE = "Milk_Yield_L"

DROP_FEATURES = [
    "Cattle_ID", "Farm_ID", "Feed_Quantity_lb", "Climate_Zone",
    "Management_System", "Feed_Type", "Feeding_Frequency",
    "Walking_Distance_km", "Grazing_Duration_hrs", "Rumination_Time_hrs",
    "Resting_Hours", "Humidity_percent", "BVD_Vaccine", "FMD_Vaccine",
    "Brucellosis_Vaccine", "HS_Vaccine", "BQ_Vaccine", "Housing_Score",
    "Body_Condition_Score", "Milking_Interval_hrs", "Breed",
]

CATEGORICAL_FEATURES = ["Date", "Young", "Lactation_Stage"]

STANDARD_SCALED_FEATURES = [
    "Feed_Quantity_kg", "Water_Intake_L", "Parity",
    "Ambient_Temperature_C", "Previous_Week_Avg_Yield",
    "Days_in_Milk", "Age_Months", "Weight_kg",
]

def preprocess(dtrain, dtest, scaler=None):
    """Preprocess training and test data"""
    # Convert month to season
    def month_to_season(m):
        if m in [12, 1, 2]:
            return "Winter"
        elif m in [3, 4, 5]:
            return "Spring"
        elif m in [6, 7, 8]:
            return "Summer"
        else:
            return "Fall"

    for df in [dtrain, dtest]:
        months = pd.to_datetime(df['Date']).dt.month
        df.drop(columns=['Date'], inplace=True)
        df['Date'] = months.apply(month_to_season)
        df['Young'] = (df['Age_Months'] < 60).astype(int)

    # Imputation
    median_val = dtrain["Feed_Quantity_kg"].median()
    dtrain.loc[dtrain["Feed_Quantity_kg"].isna(), "Feed_Quantity_kg"] = median_val
    dtest.loc[dtest["Feed_Quantity_kg"].isna(), "Feed_Quantity_kg"] = median_val

    # Drop features
    dtrain = dtrain.drop(DROP_FEATURES, axis=1)
    dtest = dtest.drop(DROP_FEATURES, axis=1)

    # One-hot encode
    dtrain = pd.get_dummies(dtrain, columns=CATEGORICAL_FEATURES, drop_first=True)
    dtest = pd.get_dummies(dtest, columns=CATEGORICAL_FEATURES, drop_first=True)
    dtrain, dtest = dtrain.align(dtest, join='left', axis=1, fill_value=0)

    # Standardize
    if scaler is None:
        scaler = StandardScaler()
        dtrain[STANDARD_SCALED_FEATURES] = scaler.fit_transform(dtrain[STANDARD_SCALED_FEATURES])
    else:
        dtrain[STANDARD_SCALED_FEATURES] = scaler.transform(dtrain[STANDARD_SCALED_FEATURES])
    
    dtest[STANDARD_SCALED_FEATURES] = scaler.transform(dtest[STANDARD_SCALED_FEATURES])

    return dtrain, dtest, scaler

# Load data and create validation split
print("Loading data and creating validation split...")
train_data = pd.read_csv(TRAIN_PATH)

# Split: 60% train, 20% dev (for finding optimal iters), 20% validation (for final eval)
X_temp, X_val, y_temp, y_val = train_test_split(
    train_data.drop(TARGET_FEATURE, axis=1),
    train_data[TARGET_FEATURE], 
    test_size=0.2, 
    random_state=42
)

X_train, X_dev, y_train, y_dev = train_test_split(
    X_temp, y_temp, 
    test_size=0.25,  # 0.25 of 80% = 20% of total
    random_state=42
)

print(f"Train size: {len(X_train)}, Dev size: {len(X_dev)}, Validation size: {len(X_val)}")

# Preprocess
X_train_proc, X_dev_proc, scaler = preprocess(X_train.copy(), X_dev.copy())
_, X_val_proc, _ = preprocess(X_train.copy(), X_val.copy(), scaler=scaler)

# Model template
model_template = MLPRegressor(
    hidden_layer_sizes=(110, 110, 110),
    activation="tanh",
    learning_rate_init=0.00003,
    learning_rate="adaptive",
    early_stopping=False,
    n_iter_no_change=20,
    verbose=False,
    warm_start=True,
    max_iter=10,
    random_state=1
)

TOLERANCE = 0.00005

Loading data and creating validation split...
Train size: 126000, Dev size: 42000, Validation size: 42000


In [2]:
# Find optimal iterations using train/dev split
print("\n" + "="*60)
print("FINDING OPTIMAL ITERATIONS")
print("="*60)

model = clone(model_template)
prev_rmse = float('inf')
dev_rmse = 0
total_iterations = 0

while dev_rmse + TOLERANCE < prev_rmse:
    if dev_rmse > 0:
        prev_rmse = dev_rmse
    
    model.fit(X_train_proc, y_train)
    
    y_train_pred = model.predict(X_train_proc)
    y_dev_pred = model.predict(X_dev_proc)
    
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    dev_rmse = np.sqrt(mean_squared_error(y_dev, y_dev_pred))
    
    total_iterations += 10
    print(f"Iteration {total_iterations}: Train RMSE={train_rmse:.5f}, Dev RMSE={dev_rmse:.5f}")

print(f"\nOptimal iterations found: {total_iterations}")

# Approach 1: Single model trained on train+dev
print("\n" + "="*60)
print("APPROACH 1: SINGLE MODEL ON ALL TRAINING DATA")
print("="*60)

X_all_train = pd.concat([X_train, X_dev])
y_all_train = pd.concat([y_train, y_dev])

X_all_train_proc, X_val_proc_1, scaler_1 = preprocess(X_all_train.copy(), X_val.copy())

single_model = clone(model_template)
single_model.max_iter = total_iterations
single_model.fit(X_all_train_proc, y_all_train)

y_val_pred_single = single_model.predict(X_val_proc_1)
val_rmse_single = np.sqrt(mean_squared_error(y_val, y_val_pred_single))

print(f"Single Model - Validation RMSE: {val_rmse_single:.5f}")


FINDING OPTIMAL ITERATIONS
Iteration 10: Train RMSE=4.19483, Dev RMSE=4.16969
Iteration 20: Train RMSE=4.13134, Dev RMSE=4.10543
Iteration 30: Train RMSE=4.11857, Dev RMSE=4.09475
Iteration 40: Train RMSE=4.11334, Dev RMSE=4.09076
Iteration 50: Train RMSE=4.11040, Dev RMSE=4.08866
Iteration 60: Train RMSE=4.10840, Dev RMSE=4.08738
Iteration 70: Train RMSE=4.10690, Dev RMSE=4.08655
Iteration 80: Train RMSE=4.10570, Dev RMSE=4.08602
Iteration 90: Train RMSE=4.10472, Dev RMSE=4.08571
Iteration 100: Train RMSE=4.10388, Dev RMSE=4.08556
Iteration 110: Train RMSE=4.10313, Dev RMSE=4.08551

Optimal iterations found: 110

APPROACH 1: SINGLE MODEL ON ALL TRAINING DATA
Single Model - Validation RMSE: 4.12715


In [3]:
# Approach 2: 5-fold ensemble + Full data model
print("\n" + "="*60)
print("APPROACH 2: HYBRID ENSEMBLE (5 K-FOLD + 1 FULL)")
print("="*60)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_models = []
fold_scalers = []
fold_iterations = []
fold_val_rmses = []  # Track validation RMSEs for weighting

X_all_train = pd.concat([X_train, X_dev])
y_all_train = pd.concat([y_train, y_dev])
X_all_train_reset = X_all_train.reset_index(drop=True)
y_all_train_reset = y_all_train.reset_index(drop=True)

# Train K-Fold models
for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_all_train_reset)):
    print(f"\nTraining fold {fold_idx + 1}/5...")
    
    # Split fold into train and internal validation for early stopping
    X_fold_train = X_all_train_reset.iloc[train_idx]
    y_fold_train = y_all_train_reset.iloc[train_idx]
    X_fold_val = X_all_train_reset.iloc[val_idx]
    y_fold_val = y_all_train_reset.iloc[val_idx]
    
    # Preprocess fold data
    X_fold_train_proc, X_fold_val_proc, fold_scaler = preprocess(
        X_fold_train.copy(), 
        X_fold_val.copy()
    )
    
    # Find optimal iterations for this fold using tolerance method
    fold_model = clone(model_template)
    prev_fold_rmse = float('inf')
    fold_val_rmse = 0
    fold_iters = 0
    
    while fold_val_rmse + TOLERANCE < prev_fold_rmse:
        if fold_val_rmse > 0:
            prev_fold_rmse = fold_val_rmse
        
        fold_model.fit(X_fold_train_proc, y_fold_train)
        
        y_fold_val_pred = fold_model.predict(X_fold_val_proc)
        fold_val_rmse = np.sqrt(mean_squared_error(y_fold_val, y_fold_val_pred))
        
        fold_iters += 10
        if fold_iters % 50 == 0:  # Print every 50 iterations
            print(f"  Iteration {fold_iters}: Fold Val RMSE={fold_val_rmse:.5f}")
    
    print(f"  Optimal iterations for fold {fold_idx + 1}: {fold_iters}")
    fold_iterations.append(fold_iters)
    
    fold_models.append(fold_model)
    fold_scalers.append(fold_scaler)
    
    # Evaluate on held-out validation set
    _, X_val_proc_fold, _ = preprocess(
        X_fold_train.copy(), 
        X_val.copy(),
        scaler=fold_scaler
    )
    y_fold_pred = fold_model.predict(X_val_proc_fold)
    fold_rmse = np.sqrt(mean_squared_error(y_val, y_fold_pred))
    fold_val_rmses.append(fold_rmse)
    print(f"  Fold {fold_idx + 1} on True Validation RMSE: {fold_rmse:.5f}")

# Train full-data model
print("\n" + "="*60)
print("Training full-data model (6th ensemble member)...")
print("="*60)

avg_iters = int(np.mean(fold_iterations))
print(f"Using average iterations from folds: {avg_iters}")

X_all_train_proc, X_val_proc_full, full_scaler = preprocess(
    X_all_train.copy(),
    X_val.copy()
)

full_model = clone(model_template)
full_model.max_iter = avg_iters
full_model.fit(X_all_train_proc, y_all_train)

y_val_pred_full = full_model.predict(X_val_proc_full)
full_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_full))
print(f"Full model - Validation RMSE: {full_rmse:.5f}")

# Add full model to ensemble
fold_models.append(full_model)
fold_scalers.append(full_scaler)
fold_val_rmses.append(full_rmse)


APPROACH 2: HYBRID ENSEMBLE (5 K-FOLD + 1 FULL)

Training fold 1/5...
  Iteration 50: Fold Val RMSE=4.11732
  Iteration 100: Fold Val RMSE=4.11554
  Optimal iterations for fold 1: 100
  Fold 1 on True Validation RMSE: 4.12940

Training fold 2/5...
  Iteration 50: Fold Val RMSE=4.12055
  Iteration 100: Fold Val RMSE=4.11775
  Optimal iterations for fold 2: 120
  Fold 2 on True Validation RMSE: 4.12844

Training fold 3/5...
  Iteration 50: Fold Val RMSE=4.11148
  Iteration 100: Fold Val RMSE=4.10764
  Optimal iterations for fold 3: 120
  Fold 3 on True Validation RMSE: 4.12985

Training fold 4/5...
  Iteration 50: Fold Val RMSE=4.11716
  Iteration 100: Fold Val RMSE=4.11476
  Optimal iterations for fold 4: 110
  Fold 4 on True Validation RMSE: 4.12714

Training fold 5/5...


/opt/anaconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


  Iteration 50: Fold Val RMSE=4.08533
  Iteration 100: Fold Val RMSE=4.08213
  Optimal iterations for fold 5: 110
  Fold 5 on True Validation RMSE: 4.12678

Training full-data model (6th ensemble member)...
Using average iterations from folds: 112
Full model - Validation RMSE: 4.12890


In [4]:
# Ensemble predictions
print("\n" + "="*60)
print("Generating ensemble predictions...")
print("="*60)

# Get all predictions from all models
ensemble_preds_all = []
for idx in range(6):  # All 6 models
    fold_model = fold_models[idx]
    fold_scaler = fold_scalers[idx]
    
    _, X_val_proc_fold, _ = preprocess(
        X_all_train.copy(), 
        X_val.copy(), 
        scaler=fold_scaler
    )
    fold_pred = fold_model.predict(X_val_proc_fold)
    ensemble_preds_all.append(fold_pred)
    print(f"Model {idx + 1} prediction mean: {fold_pred.mean():.5f}, Val RMSE: {fold_val_rmses[idx]:.5f}")

# Simple average (baseline)
y_val_pred_simple = np.mean(ensemble_preds_all, axis=0)
val_rmse_simple = np.sqrt(mean_squared_error(y_val, y_val_pred_simple))

print(f"\nSimple Average Ensemble - Validation RMSE: {val_rmse_simple:.5f}")


Generating ensemble predictions...
Model 1 prediction mean: 15.44723, Val RMSE: 4.12940
Model 2 prediction mean: 15.53494, Val RMSE: 4.12844
Model 3 prediction mean: 15.48566, Val RMSE: 4.12985
Model 4 prediction mean: 15.63782, Val RMSE: 4.12714
Model 5 prediction mean: 15.58323, Val RMSE: 4.12678
Model 6 prediction mean: 15.46004, Val RMSE: 4.12890

Simple Average Ensemble - Validation RMSE: 4.12691


In [5]:
# Weighted average based on inverse RMSE
print("\n" + "="*60)
print("WEIGHTED ENSEMBLE (Inverse RMSE)")
print("="*60)

weights = np.array([1/rmse for rmse in fold_val_rmses])
weights = weights / weights.sum()  # Normalize to sum to 1

print("Model weights:")
for idx, (w, rmse) in enumerate(zip(weights, fold_val_rmses)):
    print(f"  Model {idx + 1}: weight={w:.4f}, RMSE={rmse:.5f}")

y_val_pred_weighted = np.average(ensemble_preds_all, axis=0, weights=weights)
val_rmse_weighted = np.sqrt(mean_squared_error(y_val, y_val_pred_weighted))

print(f"\nWeighted Ensemble - Validation RMSE: {val_rmse_weighted:.5f}")


WEIGHTED ENSEMBLE (Inverse RMSE)
Model weights:
  Model 1: weight=0.1666, RMSE=4.12940
  Model 2: weight=0.1667, RMSE=4.12844
  Model 3: weight=0.1666, RMSE=4.12985
  Model 4: weight=0.1667, RMSE=4.12714
  Model 5: weight=0.1667, RMSE=4.12678
  Model 6: weight=0.1666, RMSE=4.12890

Weighted Ensemble - Validation RMSE: 4.12691


In [6]:
# Residual Stacking
print("\n" + "="*60)
print("RESIDUAL STACKING")
print("="*60)

# Get predictions on training data for residual calculation
train_preds_for_residuals = []
for idx in range(6):
    fold_model = fold_models[idx]
    fold_scaler = fold_scalers[idx]
    
    X_all_train_proc_temp, _, _ = preprocess(
        X_all_train.copy(), 
        X_all_train.copy(),  # Use same data for both args
        scaler=fold_scaler
    )
    train_pred = fold_model.predict(X_all_train_proc_temp)
    train_preds_for_residuals.append(train_pred)

# Calculate residuals from weighted ensemble on training data
y_train_pred_weighted = np.average(train_preds_for_residuals, axis=0, weights=weights)
residuals = y_all_train.values - y_train_pred_weighted

print(f"Training residuals - Mean: {residuals.mean():.5f}, Std: {residuals.std():.5f}")

# Train residual model (using Ridge regression for stability)
residual_model = Ridge(alpha=10.0, random_state=42)
residual_model.fit(X_all_train_proc, residuals)

# Predict residuals on validation set
residual_pred = residual_model.predict(X_val_proc_full)
print(f"Validation residual predictions - Mean: {residual_pred.mean():.5f}, Std: {residual_pred.std():.5f}")

# Apply residual correction with dampening factor
dampening_factors = [0.3, 0.5, 0.7, 1.0]
best_dampening = 0
best_stacked_rmse = float('inf')

print("\nTesting dampening factors:")
for factor in dampening_factors:
    y_val_pred_stacked = y_val_pred_weighted + factor * residual_pred
    stacked_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_stacked))
    print(f"  Dampening {factor:.1f}: RMSE = {stacked_rmse:.5f}")
    
    if stacked_rmse < best_stacked_rmse:
        best_stacked_rmse = stacked_rmse
        best_dampening = factor

print(f"\nBest dampening factor: {best_dampening}")
y_val_pred_stacked = y_val_pred_weighted + best_dampening * residual_pred
val_rmse_stacked = best_stacked_rmse


RESIDUAL STACKING
Training residuals - Mean: 0.06562, Std: 4.09676
Validation residual predictions - Mean: 0.06566, Std: 0.02095

Testing dampening factors:
  Dampening 0.3: RMSE = 4.12663
  Dampening 0.5: RMSE = 4.12650
  Dampening 0.7: RMSE = 4.12642
  Dampening 1.0: RMSE = 4.12639

Best dampening factor: 1.0


In [7]:
# Residual Stacking
print("\n" + "="*60)
print("RESIDUAL STACKING - COMPARISON")
print("="*60)

# ============================================================
# APPROACH 1: Train residual model on 5 K-FOLD MODELS ONLY
# ============================================================
print("\nAPPROACH 1: Residual model trained on 5 K-Fold models only")
print("-" * 60)

# Get predictions on training data from ONLY 5 fold models
train_preds_5fold = []
for idx in range(5):  # Only first 5 models
    fold_model = fold_models[idx]
    fold_scaler = fold_scalers[idx]
    
    X_all_train_proc_temp, _, _ = preprocess(
        X_all_train.copy(), 
        X_all_train.copy(),
        scaler=fold_scaler
    )
    train_pred = fold_model.predict(X_all_train_proc_temp)
    train_preds_5fold.append(train_pred)

# Calculate weights for 5-fold models only
weights_5fold = np.array([1/rmse for rmse in fold_val_rmses[:5]])
weights_5fold = weights_5fold / weights_5fold.sum()

print("5-Fold Model weights:")
for idx, (w, rmse) in enumerate(zip(weights_5fold, fold_val_rmses[:5])):
    print(f"  Fold {idx + 1}: weight={w:.4f}, Val RMSE={rmse:.5f}")

# Calculate residuals from 5-fold weighted ensemble
y_train_pred_5fold = np.average(train_preds_5fold, axis=0, weights=weights_5fold)
residuals_5fold = y_all_train.values - y_train_pred_5fold

print(f"\nTraining residuals (5-fold) - Mean: {residuals_5fold.mean():.5f}, Std: {residuals_5fold.std():.5f}")

# Train residual model on 5-fold residuals
residual_model_5fold = Ridge(alpha=10.0, random_state=42)
residual_model_5fold.fit(X_all_train_proc, residuals_5fold)

# Get 5-fold validation predictions
val_preds_5fold = []
for idx in range(5):
    fold_model = fold_models[idx]
    fold_scaler = fold_scalers[idx]
    
    _, X_val_proc_fold, _ = preprocess(
        X_all_train.copy(), 
        X_val.copy(), 
        scaler=fold_scaler
    )
    val_pred = fold_model.predict(X_val_proc_fold)
    val_preds_5fold.append(val_pred)

y_val_pred_5fold_weighted = np.average(val_preds_5fold, axis=0, weights=weights_5fold)

# Predict residuals on validation set
residual_pred_5fold = residual_model_5fold.predict(X_val_proc_full)
print(f"Validation residual predictions (5-fold) - Mean: {residual_pred_5fold.mean():.5f}, Std: {residual_pred_5fold.std():.5f}")

# Test dampening factors for 5-fold
dampening_factors = [0.3, 0.5, 0.7, 1.0]
best_dampening_5fold = 0
best_stacked_rmse_5fold = float('inf')

print("\nTesting dampening factors (5-fold):")
for factor in dampening_factors:
    y_val_pred_stacked = y_val_pred_5fold_weighted + factor * residual_pred_5fold
    stacked_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_stacked))
    print(f"  Dampening {factor:.1f}: RMSE = {stacked_rmse:.5f}")
    
    if stacked_rmse < best_stacked_rmse_5fold:
        best_stacked_rmse_5fold = stacked_rmse
        best_dampening_5fold = factor

print(f"\nBest dampening factor (5-fold): {best_dampening_5fold}")
y_val_pred_stacked_5fold = y_val_pred_5fold_weighted + best_dampening_5fold * residual_pred_5fold
val_rmse_stacked_5fold = best_stacked_rmse_5fold

# ============================================================
# APPROACH 2: Train residual model on ALL 6 MODELS
# ============================================================
print("\n" + "="*60)
print("APPROACH 2: Residual model trained on all 6 models (5 folds + full)")
print("-" * 60)

# Get predictions on training data for ALL 6 models
train_preds_for_residuals = []
for idx in range(6):
    fold_model = fold_models[idx]
    fold_scaler = fold_scalers[idx]
    
    X_all_train_proc_temp, _, _ = preprocess(
        X_all_train.copy(), 
        X_all_train.copy(),
        scaler=fold_scaler
    )
    train_pred = fold_model.predict(X_all_train_proc_temp)
    train_preds_for_residuals.append(train_pred)

print("All 6 model weights:")
for idx, (w, rmse) in enumerate(zip(weights, fold_val_rmses)):
    model_type = "Fold" if idx < 5 else "Full"
    print(f"  {model_type} {idx + 1}: weight={w:.4f}, Val RMSE={rmse:.5f}")

# Calculate residuals from 6-model weighted ensemble
y_train_pred_weighted = np.average(train_preds_for_residuals, axis=0, weights=weights)
residuals = y_all_train.values - y_train_pred_weighted

print(f"\nTraining residuals (6 models) - Mean: {residuals.mean():.5f}, Std: {residuals.std():.5f}")

# Train residual model on 6-model residuals
residual_model = Ridge(alpha=10.0, random_state=42)
residual_model.fit(X_all_train_proc, residuals)

# Predict residuals on validation set
residual_pred = residual_model.predict(X_val_proc_full)
print(f"Validation residual predictions (6 models) - Mean: {residual_pred.mean():.5f}, Std: {residual_pred.std():.5f}")

# Test dampening factors for 6-model
best_dampening = 0
best_stacked_rmse = float('inf')

print("\nTesting dampening factors (6 models):")
for factor in dampening_factors:
    y_val_pred_stacked = y_val_pred_weighted + factor * residual_pred
    stacked_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred_stacked))
    print(f"  Dampening {factor:.1f}: RMSE = {stacked_rmse:.5f}")
    
    if stacked_rmse < best_stacked_rmse:
        best_stacked_rmse = stacked_rmse
        best_dampening = factor

print(f"\nBest dampening factor (6 models): {best_dampening}")
y_val_pred_stacked = y_val_pred_weighted + best_dampening * residual_pred
val_rmse_stacked = best_stacked_rmse

# ============================================================
# COMPARISON
# ============================================================
print("\n" + "="*60)
print("RESIDUAL STACKING COMPARISON")
print("="*60)
print(f"5-Fold Weighted Ensemble:              {np.sqrt(mean_squared_error(y_val, y_val_pred_5fold_weighted)):.5f}")
print(f"5-Fold + Residual Stacking:            {val_rmse_stacked_5fold:.5f}")
print(f"6-Model Weighted Ensemble:             {val_rmse_weighted:.5f}")
print(f"6-Model + Residual Stacking:           {val_rmse_stacked:.5f}")

print("\n" + "="*60)
print("IMPROVEMENTS FROM RESIDUAL STACKING")
print("="*60)
improvement_5fold = np.sqrt(mean_squared_error(y_val, y_val_pred_5fold_weighted)) - val_rmse_stacked_5fold
improvement_6model = val_rmse_weighted - val_rmse_stacked
print(f"5-Fold approach improvement:           {improvement_5fold:.5f}")
print(f"6-Model approach improvement:          {improvement_6model:.5f}")

if val_rmse_stacked_5fold < val_rmse_stacked:
    print(f"\n✓ 5-Fold residual stacking is better by: {val_rmse_stacked - val_rmse_stacked_5fold:.5f}")
else:
    print(f"\n✓ 6-Model residual stacking is better by: {val_rmse_stacked_5fold - val_rmse_stacked:.5f}")


RESIDUAL STACKING - COMPARISON

APPROACH 1: Residual model trained on 5 K-Fold models only
------------------------------------------------------------
5-Fold Model weights:
  Fold 1: weight=0.1999, Val RMSE=4.12940
  Fold 2: weight=0.2000, Val RMSE=4.12844
  Fold 3: weight=0.1999, Val RMSE=4.12985
  Fold 4: weight=0.2001, Val RMSE=4.12714
  Fold 5: weight=0.2001, Val RMSE=4.12678

Training residuals (5-fold) - Mean: 0.05266, Std: 4.09701
Validation residual predictions (5-fold) - Mean: 0.05269, Std: 0.02266

Testing dampening factors (5-fold):
  Dampening 0.3: RMSE = 4.12658
  Dampening 0.5: RMSE = 4.12650
  Dampening 0.7: RMSE = 4.12645
  Dampening 1.0: RMSE = 4.12644

Best dampening factor (5-fold): 1.0

APPROACH 2: Residual model trained on all 6 models (5 folds + full)
------------------------------------------------------------
All 6 model weights:
  Fold 1: weight=0.1666, Val RMSE=4.12940
  Fold 2: weight=0.1667, Val RMSE=4.12844
  Fold 3: weight=0.1666, Val RMSE=4.12985
  Fold